In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Load data
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

# Create FamilySize
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

# Extract Title from Name
train["Title"] = train["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Simplify titles
rare_titles = ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"]
train["Title"] = train["Title"].replace(rare_titles, "Rare")
test["Title"] = test["Title"].replace(rare_titles, "Rare")

train["Title"] = train["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
test["Title"] = test["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})

# Map categorical to numbers
title_map = {"Mr":0, "Miss":1, "Mrs":2, "Master":3, "Rare":4}
train["Title"] = train["Title"].map(title_map)
test["Title"] = test["Title"].map(title_map)

train["Sex"] = train["Sex"].map({"male":0, "female":1})
test["Sex"] = test["Sex"].map({"male":0, "female":1})

# Fill missing values
train["Age"] = train["Age"].fillna(train["Age"].median())
test["Age"] = test["Age"].fillna(test["Age"].median())
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

train["Embarked"] = train["Embarked"].fillna("S").map({"S":0,"C":1,"Q":2})
test["Embarked"] = test["Embarked"].fillna("S").map({"S":0,"C":1,"Q":2})

features = ["Pclass","Sex","Age","Fare","Embarked","FamilySize","Title"]

X = train[features]
y = train["Survived"]

# Train model (tuned)
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=1
)

model.fit(X, y)

predictions = model.predict(test[features])

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
